# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [2]:
df["quality rating"].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [3]:
y = (df["quality rating"] >= 6).astype(int)
y.head()

0    1
1    1
2    0
3    1
4    0
Name: quality rating, dtype: int64

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [4]:
y.value_counts(), y.value_counts(normalize=True)

(quality rating
 0    50010
 1    49990
 Name: count, dtype: int64,
 quality rating
 0    0.5001
 1    0.4999
 Name: proportion, dtype: float64)

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [5]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["quality rating"])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
X_scaled.shape, y.shape

((100000, 10), (100000,))

## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [7]:
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

results = []

for solver in solvers:
    model = LogisticRegression(
        solver=solver,
        max_iter=1000,
        n_jobs=-1
    )
    
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "solver": solver,
        "fit_time_sec": fit_time,
        "accuracy": acc
    })

results_df = pd.DataFrame(results).sort_values("fit_time_sec")
results_df

/Users/gizemtotkanli/.pyenv/versions/3.12.9/envs/workintech_current/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 14.
  warnings.warn(


,solver,fit_time_sec,accuracy
2,liblinear,0.063733,0.8613
3,sag,0.317669,0.8613
4,saga,0.597901,0.8613
1,lbfgs,0.761978,0.8612
0,newton-cg,0.976174,0.8613


In [8]:
fastest_solver = results_df.iloc[0]["solver"]
fastest_solver

'liblinear'

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [9]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/gizemtotkanli/.pyenv/versions/3.12.9/envs/workintech_current/bin/python
cachedir: .pytest_cache
rootdir: /Users/gizemtotkanli/code/totkanligizem/S16D4-S-data-solvers/tests
plugins: dash-3.3.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.00s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
import time

# SGD Logistic Regression
sgd_clf = SGDClassifier(
    loss="log_loss",     # logistic regression
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

start_time = time.time()
sgd_clf.fit(X, y)
sgd_fit_time = time.time() - start_time

y_pred_sgd = sgd_clf.predict(X)
sgd_accuracy = accuracy_score(y, y_pred_sgd)

sgd_fit_time, sgd_accuracy

(0.366854190826416, 0.8584)

☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [11]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# features
X_raw = df.drop(columns="quality rating")
y = (df["quality rating"] >= 6).astype(int)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

best_model = LogisticRegression(
    solver="liblinear",
    penalty="l1",
    C=1.0,
    max_iter=1000,
    random_state=42
)
best_model.fit(X, y)

new_wine_scaled = scaler.transform(new_wine)

predicted_class = int(best_model.predict(new_wine_scaled)[0])
predicted_proba_of_class = float(best_model.predict_proba(new_wine_scaled)[0][predicted_class])

predicted_class, predicted_proba_of_class

(0, 0.9686678026280313)

# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [17]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/gizemtotkanli/.pyenv/versions/3.12.9/envs/workintech_current/bin/python
cachedir: .pytest_cache
rootdir: /Users/gizemtotkanli/code/totkanligizem/S16D4-S-data-solvers/tests
plugins: dash-3.3.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.00s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master



## 📌 Sonuç ve Değerlendirme – Lojistik Regresyon Çözücüleri

Bu çalışmada, Lojistik Regresyon modellerinde kullanılan farklı çözücülerin (**newton-cg, lbfgs, liblinear, sag, saga**) model performansı ve yakınsama süresi üzerindeki etkileri incelenmiştir.

### 🔍 Solver Karşılaştırması
- Tüm çözücüler benzer **accuracy** değerleri üretmiştir (~0.86).  
- Bu durum, problemin maliyet fonksiyonunun görece **kolay ve konveks** olmasından kaynaklanmaktadır.
- **liblinear**, açık ara en kısa uyum süresine sahip çözücü olmuştur.
- Özellikle L1 regularizasyon ile birlikte kullanıldığında, gereksiz özellikleri sıfırlayarak bu veri seti için avantaj sağlamaktadır.

### ⚡ SGD ile Lojistik Regresyon
- `SGDClassifier` ile eğitilen model, benzer doğruluk seviyesine çok daha kısa sürede ulaşmıştır.
- Bu hız kazanımı, Stokastik Gradyan İnişi’nin her adımda tüm veri yerine **tekil örnekler** üzerinden güncelleme yapmasından kaynaklanmaktadır.
- Büyük veri setleri için ölçeklenebilir bir çözüm sunduğu gözlemlenmiştir.

### 🍷 Yeni Şarap Tahmini
En dengeli model olarak seçilen **liblinear + L1** kombinasyonu ile yeni şarap için yapılan tahmin:

- **Tahmin edilen sınıf:** 0 (düşük kalite)
- **Bu sınıfa ait olasılık:** **0.9687**

Model, tahmininde yüksek bir güven seviyesine sahiptir ve test gereksinimlerini başarıyla karşılamıştır.

### 🧠 Genel Yorum
- Varsayılan solver olan **lbfgs**, çoğu durumda iyi bir başlangıç noktasıdır.
- Ancak bu veri seti özelinde, **liblinear** hem hız hem de yorumlanabilirlik açısından daha uygun bir tercih olmuştur.
- Solver seçimi; veri boyutu, özellik sayısı ve regularizasyon ihtiyacına göre mutlaka değerlendirilmelidir.